# Red Wine Quality Prediction

This project builds a binary classification model to identify high-quality red wines from physicochemical measurements. The original `quality` score is converted into a portfolio-friendly target:

- `1`: high-quality wine, score >= 7
- `0`: standard-quality wine, score < 7

The workflow covers data validation, exploratory data analysis, model comparison, and interpretation of the best baseline model.

## Project Goals

1. Explore which chemical properties are associated with higher wine quality.
2. Train multiple classification models using reproducible scikit-learn pipelines.
3. Compare models with metrics that account for class imbalance, not accuracy alone.
4. Summarize insights that would be useful for a GitHub portfolio or CV project description.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='Set2')

## Load Data

The dataset used here is the UCI red wine quality dataset, commonly distributed on Kaggle as `winequality-red.csv`. Keep the CSV in `data/winequality-red.csv` so the notebook runs without Colab-specific paths.

In [ ]:
DATA_PATH = Path('..') / 'data' / 'winequality-red.csv'

df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().to_frame('missing_values')

In [ ]:
df.describe().T

## Target Engineering

The raw wine score ranges from 3 to 8 in this dataset. Since high scores are much less common, this is an imbalanced binary classification problem.

In [ ]:
quality_counts = df['quality'].value_counts().sort_index()
quality_counts

In [ ]:
model_df = df.copy()
model_df['good_quality'] = (model_df['quality'] >= 7).astype(int)
model_df = model_df.drop(columns='quality')

class_balance = model_df['good_quality'].value_counts(normalize=True).rename('share')
class_balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=model_df, x='good_quality', hue='good_quality', legend=False, ax=ax)
ax.set_title('Class Balance: Standard vs High-Quality Wine')
ax.set_xlabel('High-quality wine')
ax.set_ylabel('Number of wines')
ax.set_xticklabels(['Standard (< 7)', 'High quality (>= 7)'])
plt.tight_layout()
plt.show()

## Exploratory Data Analysis

The strongest individual relationships with high quality are often expected around alcohol, sulphates, volatile acidity, and citric acid. The plots below check those patterns before modeling.

In [ ]:
correlations = (
    model_df.corr(numeric_only=True)['good_quality']
    .drop('good_quality')
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 5))
correlations.plot(kind='barh', ax=ax, color=np.where(correlations > 0, '#2ca25f', '#de2d26'))
ax.set_title('Feature Correlation with High-Quality Label')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(model_df.corr(numeric_only=True), cmap='vlag', center=0, linewidths=0.4, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = ['alcohol', 'sulphates', 'volatile acidity', 'citric acid']
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for feature, ax in zip(features_to_plot, axes.ravel()):
    sns.boxplot(data=model_df, x='good_quality', y=feature, hue='good_quality', legend=False, ax=ax)
    ax.set_xlabel('High-quality wine')
    ax.set_xticklabels(['Standard', 'High quality'])
    ax.set_title(feature.title())

plt.tight_layout()
plt.show()

## Train/Test Split

A stratified split keeps the same high-quality/standard-quality ratio in the train and test sets. This matters because the positive class is relatively small.

In [ ]:
X = model_df.drop(columns='good_quality')
y = model_df['good_quality']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Test positive rate: {y_test.mean():.2%}')

## Model Comparison

Accuracy can look strong even when a model misses many high-quality wines. For that reason, this notebook compares accuracy, precision, recall, F1 score, and ROC-AUC. Scaled pipelines are used for distance- and margin-based models.

In [ ]:
models = {
    'Baseline - Most Frequent': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE)),
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=RANDOM_STATE),
    'K-Nearest Neighbors': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=7)),
    ]),
}

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_rows = []

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        'model': name,
        'accuracy': scores['test_accuracy'].mean(),
        'precision': scores['test_precision'].mean(),
        'recall': scores['test_recall'].mean(),
        'f1': scores['test_f1'].mean(),
        'roc_auc': scores['test_roc_auc'].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values('f1', ascending=False)
cv_results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cv_results, x='f1', y='model', ax=ax, color='#4c78a8')
ax.set_title('Cross-Validated F1 Score by Model')
ax.set_xlabel('Mean F1 score')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Final Evaluation

The top cross-validated model is refit on the full training set and evaluated once on the holdout test set.

In [ ]:
best_model_name = cv_results.iloc[0]['model']
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

if hasattr(best_model, 'predict_proba'):
    y_score = best_model.predict_proba(X_test)[:, 1]
else:
    y_score = best_model.decision_function(X_test)

test_metrics = pd.DataFrame([
    {
        'model': best_model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_score),
    }
])

test_metrics

In [ ]:
print(classification_report(y_test, y_pred, target_names=['standard', 'high_quality']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['Standard', 'High quality'],
    cmap='Blues',
    ax=ax,
    colorbar=False,
)
ax.set_title(f'Confusion Matrix: {best_model_name}')
plt.tight_layout()
plt.show()

## Interpretation

For a simple baseline project, the model comparison is more important than chasing a single accuracy number. A strong portfolio takeaway is that the dataset is imbalanced: models can reach high accuracy by predicting mostly standard-quality wine, but recall and F1 reveal whether high-quality wines are actually being detected.

Typical feature patterns to discuss:

- Higher alcohol and sulphates tend to be associated with high-quality wines.
- Higher volatile acidity tends to be associated with lower quality.
- Class imbalance makes F1 score and recall more informative than accuracy alone.

## Next Steps

- Tune decision thresholds to match the business goal, such as maximizing recall for high-quality wines.
- Add hyperparameter search for SVM, logistic regression, and tree-based models.
- Try ensemble models such as Random Forest or Gradient Boosting.
- Package the best model into a small prediction script or Streamlit demo for a stronger GitHub project.